# 05 · Calibration — recover parameters from observed data

The workflows so far *optimized* an objective (profit). Calibration is the
inverse problem — the one polarisopt exists for: given **observed data**, find
simulator inputs that reproduce it. In the real POLARIS workflow that means
matching link counts via the `link_moe` metric; here we match three taxi-system
observables with taxidemo's own `output_match` metric — a custom `Metric` plugin
registered through the `polarisopt.metrics` entry point, exactly like the `taxi`
simulator plugin.

The experiment is a **parameter-recovery test**, the standard way to validate
a calibration pipeline before trusting it on real data:

1. Pick "true" parameters and simulate synthetic field data from them
   (with seeds the calibration never sees).
2. Hide the true parameters; calibrate against the field data alone.
3. Compare what the calibration recovers to the truth.

In [ ]:
import json
from pathlib import Path

TRUE_PARAMS = {"taxi_count": 40, "base_fare": 7.5, "cost_per_tile": 8.0, "max_multiplier": 1.5}
OBSERVED_KEYS = ["journeys_completed", "pick_up_time", "missed"]

from taxidemo.runner import evaluate

# Synthetic field data: 5 episodes at the true parameters, seed block 2026
# (disjoint from the study's base_seed=42 blocks).
field = evaluate(TRUE_PARAMS, seed=2026, n_repeats=5)
targets = {k: field[k] for k in OBSERVED_KEYS}

targets_path = Path.home() / "taxidemo-runs" / "calibration-targets.json"
targets_path.parent.mkdir(parents=True, exist_ok=True)
targets_path.write_text(json.dumps(targets, indent=2))
targets

## The study

[`studies/calibrate-local.yaml`](../studies/calibrate-local.yaml) is workflow
3's sequential phase pointed at a different objective: `metric: output_match`
scores each sample by the mean squared relative error against the targets
(0 = perfect match), and the phase **minimizes** it. The `epsilon` criterion
stops the loop once the discrepancy drops below 0.02 — no point paying for
iterations after the data is matched to within noise.

In [ ]:
STUDY = Path("../studies/calibrate-local.yaml")
print(STUDY.read_text())

In [ ]:
from polarisopt.studies.runner import run_study
from polarisopt.utils.plugins import load_external_plugins

load_external_plugins()
samples = run_study(STUDY)
print(f"{len(samples)} samples, statuses: {set(s.status for s in samples)}")

## Convergence of the discrepancy

In [ ]:
import os
import pandas as pd
from polarisopt.samples.store import SampleStore
from polarisopt.utils.paths import workspace_layout

layout = workspace_layout(os.path.expanduser("~/taxidemo-runs/calibrate-local"))
store = SampleStore.open(layout["db"], "taxi-calibrate")
raw = store.to_dataframe()

PARAMS = list(TRUE_PARAMS)
df = pd.concat(
    [
        raw[["id", "iteration", "status"]],
        pd.DataFrame(raw["inputs"].tolist(), columns=PARAMS, index=raw.index),
    ],
    axis=1,
)
df["discrepancy"] = [m[0] if m else None for m in raw["metric"]]
df = df[df["status"] == "finished"].sort_values("id").reset_index(drop=True)
print(f"{len(df)} finished samples, {df['iteration'].max()} iterations")

In [ ]:
import matplotlib.pyplot as plt

n_warmup = (df["iteration"] == 0).sum()
evals = range(1, len(df) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(evals, df["discrepancy"].cummin(), drawstyle="steps-post", color="k", lw=2, label="best so far")
sc = ax.scatter(evals, df["discrepancy"], c=df["iteration"], cmap="viridis", zorder=3)
ax.axvline(n_warmup + 0.5, ls="--", c="gray")
ax.axhline(0.02, ls=":", c="tab:red", label="epsilon stop (0.02)")
ax.set_yscale("log")
ax.set_xlabel("evaluation")
ax.set_ylabel("discrepancy vs field data")
ax.legend(loc="upper right")
fig.colorbar(sc, ax=ax, label="iteration");

## Inside the loop: the GP surrogate and acquisition

The convergence plot above is the *output* of the sequential phase; this is the
*mechanism* behind it. At every iteration polarisopt fits a Gaussian-process
**surrogate** (`surrogates/gp.py`, a BoTorch `SingleTaskGP` with a Matérn-5/2
ARD kernel) to every `(parameters → discrepancy)` pair seen so far, then
maximizes an **acquisition function**, Expected Improvement (EI), to choose where
to evaluate next. EI balances *exploration* (sample where the GP is uncertain)
against *exploitation* (sample where the discrepancy is predicted to be low).
That is the methodology; `gp` and `qei` are just the plugins that implement it.

The objective lives in 4-D, so to *see* the surrogate we take a 1-D slice over
fleet size (`taxi_count`), holding the pricing knobs at their recovered values,
and compare the surrogate just after the LHS warm-up with the surrogate after the
full calibration. Watch the GP sharpen and EI collapse toward zero: that collapse
is the loop's own signal that nothing is left to gain, which is why it stopped
early.

In [ ]:
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.analytic import ExpectedImprovement
from polarisopt.surrogates.gp import GPSurrogate

# Parameter bounds, in the same order as PARAMS (taxi_count, base_fare, cost_per_tile, max_multiplier).
BOUNDS = [[1, 100], [0, 15], [1, 20], [1, 3]]
best = df.loc[df["discrepancy"].idxmin()]

# 1-D slice: vary fleet size, hold the pricing knobs at their recovered values.
grid = np.linspace(1, 100, 250)
fixed = {k: float(best[k]) for k in PARAMS}
slice_X = np.column_stack([grid if k == "taxi_count" else np.full_like(grid, fixed[k]) for k in PARAMS])
slice_t = torch.as_tensor(slice_X, dtype=torch.double).unsqueeze(1)  # (G, 1, d) for the acquisition


def surrogate_slice(sub):
    """Fit polarisopt's own GP on a subset of the run, then read off the
    posterior and Expected Improvement along the fleet-size slice."""
    X = sub[PARAMS].to_numpy(float)
    Y = sub["discrepancy"].to_numpy(float)[:, None]
    gp = GPSurrogate(nu=2.5, bounds=BOUNDS)            # the `gp` surrogate plugin
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gp.fit(X, Y)
        mean, var = gp.predict(slice_X)
        ei = ExpectedImprovement(gp.model, best_f=float(Y.min()), maximize=False)  # EI for minimization
        with torch.no_grad():
            ei_vals = ei(slice_t).numpy()
    return mean.ravel(), np.sqrt(var).ravel(), ei_vals


stages = [("after 16 warm-up samples", df[df["iteration"] == 0]),
          (f"after calibration ({len(df)} samples)", df)]
results = [(title, *surrogate_slice(sub)) for title, sub in stages]
ei_max = max(ei.max() for _, _, _, ei in results)

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for col, (title, mean, sd, ei_vals) in enumerate(results):
    post, acq = axes[0, col], axes[1, col]
    post.fill_between(grid, mean - 1.645 * sd, mean + 1.645 * sd, color="tab:blue", alpha=0.2, label="GP 90%")
    post.plot(grid, mean, color="tab:blue", lw=2, label="GP mean")
    post.axvline(TRUE_PARAMS["taxi_count"], color="k", ls="--", lw=1, label="true (40)")
    post.set_title(title)

    acq.fill_between(grid, 0, ei_vals, color="tab:green", alpha=0.25)
    acq.plot(grid, ei_vals, color="tab:green", lw=2)
    acq.axvline(grid[int(ei_vals.argmax())], color="tab:green", ls=":", lw=2,
                label=f"max EI @ {int(grid[int(ei_vals.argmax())])} taxis")
    acq.set_ylim(0, ei_max * 1.05)          # shared EI scale so the collapse is visible
    acq.set_xlabel("taxi_count (fleet size)")
    acq.legend(loc="upper right", fontsize=8)

axes[0, 0].set_ylabel("GP posterior:\ndiscrepancy")
axes[0, 0].legend(loc="upper right", fontsize=8)
axes[1, 0].set_ylabel("Expected\nImprovement")
fig.suptitle("The GP surrogate sharpens and Expected Improvement collapses as the loop gathers data", y=1.00)
fig.tight_layout();

## Did it recover the truth?

Two checks, in order of importance. First, the **data space**: does the
calibrated model reproduce the observations? (Validated with seeds the
calibration never used.) Second, the **parameter space**: how close are the
recovered knobs to the truth?

In [ ]:
best = df.loc[df["discrepancy"].idxmin()]
calibrated = {k: best[k] for k in PARAMS}

# Validation re-run at held-out seeds (block 9000).
val = evaluate({k: float(v) for k, v in calibrated.items()}, seed=9000, n_repeats=5)

pd.DataFrame(
    {
        "observed (field data)": targets,
        "calibrated model": {k: val[k] for k in OBSERVED_KEYS},
    }
).round(1)

In [ ]:
pd.DataFrame(
    {
        "true": TRUE_PARAMS,
        "recovered": {k: round(float(v), 2) for k, v in calibrated.items()},
    }
)

## On identifiability

Look closely at the two tables: the data-space match is good and `taxi_count`
is typically recovered almost exactly, while the pricing knobs (`base_fare`,
`max_multiplier`) can land far from the truth — *without hurting the fit*.
That is not a calibration failure; it is the observables' fault. Below the
customer-cancellation threshold, prices change only the money outputs
(`profit`), not the physics we observe (journeys, waiting, missed) — so any
price in that regime explains the field data equally well, and the GP has no
gradient to follow. The discrepancy plot and the data-space table are the
honest report card; the parameter table tells you *which* of the
data-consistent explanations the calibration picked.

This is exactly the situation in real network calibration — link counts alone
rarely pin down every behavioral parameter. The remedies are the same too:
observe more (add `profit_per_journey` to the targets and the pricing knobs
become identifiable — try it), or screen first (workflow 2b) and only
calibrate knobs the observables actually respond to.

## Scaling up

To run this calibration on Crossover instead, swap the `runner:` block for the
Slurm one in [`studies/bo-crossover.yaml`](../studies/bo-crossover.yaml) and
move `workspace:` (and `targets_file:`) to `/lcrc/...` — nothing else changes.